# FlashMind — Flashcard Quiz Agent (T26)
**CSE476 · CA1 Project 1.** Run all cells top to bottom with your API key set. Each goal prints the full multi-step trace (STEP → TOOL CALL → TOOL RESULT).

In [ ]:
# !pip install -q openai python-dotenv
import os
# os.environ['GROQ_API_KEY'] = 'gsk_...'   # or put it in .env
from agent import FlashcardAgent
agent = FlashcardAgent()

### Helpers that simulate the student (so the demo is reproducible)
They read the pending card from memory and reply with its correct answer, or with a deliberately wrong one.

In [ ]:
def correct_reply():
    return agent.memory.get_card(agent.memory.pending_id)['answer']

def wrong_reply():
    return 'I have no idea'

def show_deck():
    for c in agent.memory.all_cards():
        print(f"#{c['id']} right={c['right']} wrong={c['wrong']}  {c['question']}")

## Goal 1 — Build a deck and start a quiz
Expect: several `add_card` calls, then `quiz_me`.

In [ ]:
agent.chat('''Add these flashcards:
1. What is the time complexity of binary search? = O(log n)
2. Which data structure uses LIFO order? = stack
3. What does SQL stand for? = Structured Query Language
4. Which sorting algorithm has O(n log n) worst case and uses divide and conquer? = merge sort
5. What protocol does HTTPS use for encryption? = TLS''')
agent.chat('Quiz me!')

## Goal 2 — Get cards wrong, watch the agent re-quiz the weak ones
Expect: `grade_answer` then `quiz_me` in the same turn; the missed card has a higher `priority_score` and `reason: WEAK card`.

In [ ]:
# Miss the first question on purpose
agent.chat(wrong_reply())
# Answer the next three correctly
for _ in range(3):
    agent.chat(correct_reply())
show_deck()

In [ ]:
# Miss another, then keep answering: the weak cards should come back more often than the strong ones
agent.chat(wrong_reply())
for _ in range(4):
    agent.chat(correct_reply())
show_deck()

## Goal 3 — Memory read-back
Expect: `get_stats` call; the answer names the weak cards from earlier turns without the user repeating them.

In [ ]:
agent.chat('How am I doing so far? Which cards should I focus on?')

In [ ]:
# Proof of memory: the raw state the tools read/write
show_deck()
print('quiz clock:', agent.memory.clock, '| turns remembered:', len(agent.memory.messages))